# B1.4 · Strategic planning and agent allocation

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *AI for Security*

Builds on **[B1.3 · Threat modelling from the architecture map](https://spbreed.github.io/cyber-commons/lessons/B1.3.html)**.

| | |
|---|---|
| Open-source tooling | Semgrep OSS, CodeQL |
| Open-weight models | GLM-4.6, Llama 3.3 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


**Stage 6 — Strategic planning.** You now have a ranked threat model. This stage
decides *where to spend the analysis budget* and *which tool or agent to point at
each target*.

The default is a uniform sweep: run every rule over every file. It is simple,
and it scales cost with repository size rather than with risk — so on a large
monorepo the deep, expensive analysis gets turned off for everything, including
the parts that needed it.

Allocation makes the trade explicit. Three inputs:

- **threat rank** from stage 5,
- **historical risk** from stage 1,
- **tool fit** — rules are cheap and precise on known patterns; model review is
  expensive and finds what rules cannot express (B1.5 measures both).

The output is an assignment: which analyser runs against which boundary, with
what budget. And the honest measure of a good allocation is not coverage — it is
**threat-weighted coverage**, because covering the health endpoint thoroughly is
not an achievement.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 2 · Stage 6 — the budget, and two ways to spend it

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Target:
    name: str; file: str; threat_score: int; historical_risk: float; loc: int

TARGETS = [
 Target("load_report → execute", "src/data/reports.py", 12, 0.82, 40),
 Target("store → open",          "src/data/docs.py",     9, 0.91, 30),
 Target("render",                "src/util/render.py",   2, 0.05, 15),
 Target("health",                "src/web/health.py",    1, 0.00, 10),
 Target("upload_doc",            "src/web/handlers.py",  9, 0.30, 60),
 Target("get_report",            "src/web/handlers.py", 11, 0.30, 60),
]

ANALYSERS = {
 # name          cost/100 LOC   finds                         precision
 "grep rules":   (1,   {"CWE-798"},                          0.50),
 "taint rules":  (4,   {"CWE-89", "CWE-78", "CWE-22"},       1.00),
 "model review": (40,  {"CWE-89","CWE-78","CWE-22","CWE-863","CWE-434"}, 0.85),
}
BUDGET = 30         # arbitrary units for one CI run — deliberately tight,
                    # because an unconstrained budget hides the whole problem

def uniform(targets, analyser, budget):
    cost_per = ANALYSERS[analyser][0]
    spend, covered = 0, []
    for t in sorted(targets, key=lambda t: t.name):
        c = cost_per * t.loc / 100
        if spend + c > budget: break
        spend += c; covered.append(t)
    return {"strategy": f"uniform · {analyser}", "spend": round(spend, 1),
            "covered": covered}

def allocated(targets, budget):
    """Deep analysis on high-threat targets, cheap rules everywhere else."""
    ranked = sorted(targets, key=lambda t: -(t.threat_score + t.historical_risk * 3))
    spend, plan = 0.0, []
    for t in ranked:
        for analyser in ("model review", "taint rules", "grep rules"):
            c = ANALYSERS[analyser][0] * t.loc / 100
            wants_deep = (t.threat_score + t.historical_risk * 3) >= 9
            if analyser == "model review" and not wants_deep: continue
            if spend + c <= budget:
                spend += c; plan.append((t, analyser)); break
    return {"strategy": "allocated by threat rank", "spend": round(spend, 1),
            "plan": plan}

u = uniform(TARGETS, "model review", BUDGET)
a = allocated(TARGETS, BUDGET)
print(f"{u['strategy']:34s}spend {u['spend']:>6}  covered {len(u['covered'])}/{len(TARGETS)}")
for t in u["covered"]: print(f"      {t.name}")
print(f"\n{a['strategy']:34s}spend {a['spend']:>6}  covered {len(a['plan'])}/{len(TARGETS)}")
for t, an in a["plan"]: print(f"      {t.name:24s}{an}")

## 3 · Where it breaks — coverage is the wrong metric

In [ ]:
def coverage(plan_targets, targets):
    return len(plan_targets) / len(targets)

def threat_weighted_coverage(plan_targets, targets):
    total = sum(t.threat_score for t in targets)
    got = sum(t.threat_score for t in plan_targets)
    return got / total

u_targets = u["covered"]
a_targets = [t for t, _ in a["plan"]]

print(f"{'strategy':34s}{'coverage':>10}{'threat-weighted':>18}")
print("-" * 64)
for label, ts in (("uniform · model review", u_targets),
                  ("allocated by threat rank", a_targets)):
    print(f"{label:34s}{coverage(ts, TARGETS):>10.0%}{threat_weighted_coverage(ts, TARGETS):>18.0%}")

print("\nThe uniform sweep spent its whole budget alphabetically and covered")
print("the health endpoint before it reached the SQL sink.")
missed = [t.name for t in TARGETS if t not in u_targets and t.threat_score >= 9]
print(f"high-threat targets the uniform sweep never reached: {missed}")
assert missed

## 4 · The control — allocate, then prove the allocation was right

In [ ]:
def plan_report(plan, targets, budget):
    spend = sum(ANALYSERS[an][0] * t.loc / 100 for t, an in plan)
    covered = [t for t, _ in plan]
    deep = [t.name for t, an in plan if an == "model review"]
    uncovered_high = [t.name for t in targets
                      if t not in covered and t.threat_score >= 9]
    return {"budget": budget, "spend": round(spend, 1),
            "threat_weighted_coverage": round(threat_weighted_coverage(covered, targets), 3),
            "deep_analysis_on": deep,
            "uncovered_high_threat": uncovered_high,
            "acceptable": not uncovered_high}

r = plan_report(a["plan"], TARGETS, BUDGET)
for k, v in r.items(): print(f"{k:26s}{v}")
assert r["acceptable"]

print("\nsame budget, if someone doubles the repo with low-risk code:")
BLOAT = TARGETS + [Target(f"vendor_{i}", f"vendor/{i}.py", 1, 0.0, 200)
                   for i in range(1, 9)]
a2 = allocated(BLOAT, BUDGET)
r2 = plan_report(a2["plan"], BLOAT, BUDGET)
print(f"   threat-weighted coverage {r['threat_weighted_coverage']:.0%} → "
      f"{r2['threat_weighted_coverage']:.0%}")
print(f"   uncovered high-threat targets: {r2['uncovered_high_threat'] or 'none'}")
print("\nAllocation is what stops repository growth from silently degrading")
print("the analysis of the parts that matter.")

## What you just proved

On a tight budget the uniform model-review sweep covers only 2 of 6 targets — alphabetically, so it reaches the health endpoint before the SQL sink — giving 33% coverage but only 27% threat-weighted coverage, and missing all three high-threat targets. The allocated plan covers all six within the same budget at 100% threat-weighted coverage, puts deep model review on the SQL sink, and still holds 90% when the repository doubles in size with low-risk code.

## Your turn

Compute threat-weighted coverage for your current scanning setup. If you scan everything uniformly, the number equals your raw coverage — which means you have no allocation strategy, only a budget that will eventually be cut.

---

**Next → [B1.5 · Vulnerability auditing: three generations of SAST](https://spbreed.github.io/cyber-commons/lessons/B1.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*